# Tekne Dedektörü — Sadece Renkli Görseller — Google Colab GPU Eğitimi

Bu notebook aynı tarifle (YOLOv8s, backbone donuk `freeze=10`, imgsz=640, patience=25)
ama datasetin **sadece gerçek renkli videolardan** oluşan alt kümesiyle eğitir — gri/siyah-beyaz
kayıtlar tamamen çıkarıldı (piksel bazlı doğrulandı, sadece isme bakılmadı). Video bazlı split:
aynı videonun kareleri hem train hem val'de birden bulunmuyor.

Train: 2558, Val: 1017 (toplam 3575 görüntü, 28 video).

**Önce yapman gerekenler:**
1. Üstteki menüden **Çalışma zamanı (Runtime) > Çalışma zamanı türünü değiştir > T4 GPU** (veya A100/L4) seç.
2. `boat_color_dataset_bundle.zip` dosyasını (Mac'indeki `depth-anything` klasöründe, ~295MB) Google Drive'ına yükle.
3. Aşağıdaki hücreleri sırayla çalıştır.

In [ ]:
# 1) GPU kontrolü
!nvidia-smi

In [ ]:
# 2) Google Drive'ı bağla
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 3) Zip dosyasını Drive'dan al ve aç
# Zip'i Drive'da farklı bir yere yüklediysen ZIP_PATH'i güncelle.
ZIP_PATH = '/content/drive/MyDrive/boat_color_dataset_bundle.zip'

!rm -rf /content/work
!mkdir -p /content/work
!unzip -q "$ZIP_PATH" -d /content/work
!ls /content/work

In [ ]:
# 4) ultralytics kur
!pip install -q ultralytics

In [ ]:
# 5) dataset.yaml içindeki path'i Colab'daki yeni konuma göre düzelt
# (Mac'te: /Users/armin/Desktop/depth-anything/yolo_dataset_v4 idi)
import pathlib

yaml_path = pathlib.Path('/content/work/yolo_dataset_v4_color/dataset.yaml')
content = yaml_path.read_text()
print('--- eski ---')
print(content)

new_content = content.replace(
    '/Users/armin/Desktop/depth-anything/yolo_dataset_v4_color',
    '/content/work/yolo_dataset_v4_color'
)
yaml_path.write_text(new_content)
print('--- yeni ---')
print(yaml_path.read_text())

In [ ]:
# 6) backbone'un gercekten ilk 10 katmanda bittigini teyit et (Mac'te de dogrulanmisti)
from ultralytics import YOLO

_check = YOLO('/content/work/yolov8s.pt')
for i, layer in enumerate(_check.model.model):
    print(i, layer.__class__.__name__)

In [ ]:
# 7) Eğitim — freeze YOK (tam fine-tune), imgsz=960, batch=32, patience=25.
# tek fark: dataset artik SADECE renkli videolardan olusuyor (gri/s-b kayitlar cikarildi). Fresh baslatiyoruz
# (resume degil) ki renkli-only vs mono-only vs karisik-renk karsilastirmasi ayni recipe uzerinden temiz olsun.
from ultralytics import YOLO

model = YOLO('/content/work/yolov8s.pt')
results = model.train(
    data='/content/work/yolo_dataset_v4_color/dataset.yaml',
    epochs=80,
    imgsz=960,
    device=0,
    batch=32,
    patience=25,
    project='/content/work/runs_boat_yolo',
    name='boat_v4s_frozen_color',
    verbose=True,
)

In [ ]:
# 8) Eğitim koptuysa devam ettirmek için (7. hücre yerine bunu çalıştır):
# from ultralytics import YOLO
# model = YOLO('/content/work/runs_boat_yolo/boat_v4s_frozen_color/weights/last.pt')
# results = model.train(resume=True)

In [ ]:
# 9) Bitince: sonuçları Drive'a kopyala
!mkdir -p /content/drive/MyDrive/boat_v4s_frozen_color_results
!cp -r /content/work/runs_boat_yolo/boat_v4s_frozen_color /content/drive/MyDrive/boat_v4s_frozen_color_results/
print('Kopyalandı: Google Drive > boat_v4s_frozen_color_results > boat_v4s_frozen_color')

## Eğitim bitince Mac'ine geri alma

1. Drive'daki `boat_v4s_frozen_color_results/boat_v4s_frozen_color` klasörünü indir (`weights/best.pt` şart, geri kalanı - grafikler/results.csv - isteğe bağlı ama karşılaştırma için faydalı).
2. Bana zip'i ilet, ben `runs/detect/runs_boat_yolo/boat_v4s_frozen_color/` altına yerleştirip önceki (renkli+gri karışık) denemelerle karşılaştırırım.

**Not:** Bu run diğerlerinden küçük bir datasetle (3575 görüntü, öncekinin ~1/3'ü) eğitiliyor - epoch süresi daha kısa olacak ama daha az veriyle overfit riski de artabilir, dikkatli karşılaştır.